# NB02 – Data Transformation

**LSE ID:** 250100007

## Purpose

In NB01 I collected the raw product data and saved one JSON file per country in
the `data/raw/` folder. Those files are the raw API responses, which are nested
and not easy to analyse directly. Each file holds a list of products under a key
called `hits`, and inside every product the nutritional values sit in their own
dictionary called `nutriments`.

The goal of this notebook is to turn that raw data into a clean, flat table that
I can actually work with. For each country I go through every product in the file
and pull out the fields I need:

- the **barcode** (`code`)
- the **product name** (`product_name`)
- the **brand** (`brands`)
- the **sugar content** in grams per 100g (`sugars_100g`, which sits inside `nutriments`)
- the **country** the product was collected for

The other thing this notebook has to handle is missing data. Many products in
Open Food Facts have been scanned but never filled in, so some have no name, no
brand, or no sugar value at all.  I count how much is missing per country before removing
anything, because the amount of missing data is itself part of the picture, and
then keep only the rows that have a sugar value.

Once everything is pulled out, I save the result as a **CSV file** in the
`data/processed/` folder. That CSV becomes the input for NB03, where I do the
actual analysis (comparing sugar content between countries with and without a
sugar tax).

I do not do any analysis here. This notebook only reshapes the data from nested
JSON into a clean table.

In [1]:
import json 
import pandas as pd 
from pathlib import Path

RAW_DIR = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

rows = []
for path in RAW_DIR.glob("*.json"): 
    country = path.stem # .stem basically just takes the name of the file without the type of file  so it stores "italy"
    with open(path,  encoding='utf-8') as f: 
        sample = json.load(f)

        for reading in sample["hits"]: 
            # Some products were scanned but never filled in, so fields can be missing.
            # .get() returns None instead of crashing, and the {} and [] are fallbacks
            # so the next lines still have something to work with.
            nutriments = reading.get("nutriments", {})
            brands = reading.get("brands", [])

            rows.append({
                "country": country,
                "barcode": reading.get("code"),
                "product_name": reading.get("product_name"),
                "brand": brands[0] if brands else None, # None if the product has no brand listed.
                "sugar_content_per_100g": nutriments.get("sugars_100g")
            })

df = pd.DataFrame(rows)

In [2]:
# After generating the Data Frame, I want to have a look at the structure 
df.head()

,country,barcode,product_name,brand,sugar_content_per_100g
0,italy,0000503210227,NaN,NaN,NaN
1,italy,8002516010223,La classica,Tomarchio,11.0
2,italy,3270190005261,PULP' Saveur Orange,Carrefour,8.4
3,italy,4060800129680,Pepsi lemon,Pepsi,10.7
4,italy,4060800001771,Pepsi-cola,pepsi,10.9


### What one row represents

One row is **one product as sold in one country**. The five columns are the
barcode, the product name, the first listed brand, the sugar content in grams per
100g, and the country.

The country is the only column that does not come from the product record. Each
API request already filtered to a single country, so the response never repeats
that information product by product; I take it from the filename each set of
products was saved under and stamp it onto every row.

`brands` arrives from the API as a list, because a product can carry more than one
brand. I keep the first entry, which is the primary brand in the records I
inspected.

In [3]:
# Some products were scanned but never filled in, so sugar can be missing.
# .size counts every row, .count ignores missing values, so the gap between
# them is how many products have no sugar recorded.
# I am going to use a .groupby to figure it out 
print(df.groupby("country")["sugar_content_per_100g"].agg(["count", "size"])) 

                count  size
country                    
france           3497  3765
germany          1819  1943
italy             323   337
united-kingdom    447   512


### Filtering out the missing sugar values

Looking at the first rows of the table, the very first product has no name, no
brand and no sugar value. It is a real record in Open Food Facts: someone scanned
the barcode and never filled anything in. Records like this are common in a
database built by volunteers, so the table needs to deal with them rather than
assume every product is complete.

Before removing anything, I counted how many products each country has and how
many of those have a sugar value, using `.groupby()` with `count` and `size`.
`size` counts every row, while `count` ignores missing values, so the difference
between them is the number of products with no sugar recorded:

| Country        | Products collected | With a sugar value | Missing | % missing |
|----------------|--------------------|--------------------|---------|-----------|
| France         | 3,765              | 3,497              | 268     | 7.1%      |
| Germany        | 1,943              | 1,819              | 124     | 6.4%      |
| Italy          | 337                | 323                | 14      | 4.2%      |
| United Kingdom | 512                | 447                | 65      | 12.7%     |
| **Total**      | **6,557**          | **6,086**          | **471** | **7.2%**  |

The groups do not start out the same size. NB01 collected every soda Open Food
Facts holds for each country, so the sizes reflect how thoroughly volunteers have
catalogued each market: 3,765 French products against 337 Italian ones.

What differs beyond that is how complete the records are. The United Kingdom is
missing 12.7% of its sugar values, three times Italy's 4.2% and the worst of the
four. This matters because the United Kingdom is one of my two taxed countries.
If the British products with no sugar value are not a random selection of British
soft drinks, the average I calculate for the UK is built from a different kind of
drink than the average for Italy, and part of any difference I find would come
from that rather than from the tax.

I then filtered with `.notna()` to keep only rows where `sugar_content_per_100g`
is present. I deliberately did not drop rows with any missing value, because a
product with no name or brand is still usable as long as it has a sugar value,
and sugar is the only field the analysis needs.

Unequal group sizes are not in themselves a problem for comparing averages. A mean
does not require equal group sizes to be valid; the Italian mean of 323 values is
a perfectly good estimate, just less precise than the French mean of 3,497. 

In [4]:
# Open Food Facts stores one record per barcode, listing every country a product
# is sold in. A product sold in several of my four countries therefore appears
# once per country. How much of my data is shared this way?
print(f"Rows: {len(df)}")

# I am going to use .nunique() taken out of the Pandas Documentation to single out unique barcodes.
print(f"Unique barcodes: {df['barcode'].nunique()}")

countries_per_product = df.groupby("barcode")["country"].nunique()
print(countries_per_product.value_counts().sort_index())

Rows: 6557
Unique barcodes: 6032
country
1    5546
2     451
3      31
4       4
Name: count, dtype: int64


### Does the same product appear in more than one country?

Open Food Facts stores one record per barcode, listing every country a product is
sold in. A product sold in several of my four countries therefore appears once per
country. That would matter: identical products contribute the same sugar value to
both my taxed and untaxed averages, flattening any difference before I measure it.

My 6,557 rows come from 6,032 distinct barcodes. Of those, 5,546 (92%) appear in
only one of my four countries, 451 in two, 31 in three, and 4 in all four.

The overlap is small, so this is a check that passes. It is also worth
understanding why: a drink formulated differently for the British and Italian
markets carries a different barcode and appears as a separate record, so a shared
barcode means a genuinely identical product distributed across markets. The 92%
figure tells me most sodas in this category are market-specific products, which is
exactly the population in which reformulation would be visible.

I keep all 6,557 rows. One row is one product as sold in one country, and a
product available in two countries genuinely belongs to both country averages.

In [5]:
# I am going to create a new DataFrame that filters out all the values that are NaN for sugar contents. 
# For this, I am going to use .notna() 
df_clean = df[df["sugar_content_per_100g"].notna()]
df_clean.head()

,country,barcode,product_name,brand,sugar_content_per_100g
1,italy,8002516010223,La classica,Tomarchio,11.0
2,italy,3270190005261,PULP' Saveur Orange,Carrefour,8.4
3,italy,4060800129680,Pepsi lemon,Pepsi,10.7
4,italy,4060800001771,Pepsi-cola,pepsi,10.9
5,italy,5449000005090,Fanta,fanta,11.8


In [6]:
# Now I save it into a clean data CSV 
df_clean.to_csv(PROCESSED_DIR / "sodas_clean.csv", index=False, encoding='utf-8')

## Sources and tools

**Python libraries**

- pandas, used for building the table, grouping, filtering and writing the CSV.
  https://pandas.pydata.org/docs/

**Methods I looked up rather than took from the course**

- `Series.nunique()` — counts distinct values in a column. I used it to find how
  many unique barcodes sit behind my 6,557 rows.
  https://pandas.pydata.org/docs/reference/api/pandas.Series.nunique.html
- `Series.value_counts()` — counts how often each value appears. Applied to the
  result of `.nunique()` per barcode, it tells me how many products are sold in
  one country, two, three or four.
  https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html
- GroupBy reference, used to check that `.agg(["count", "size"])` behaves as I
  expected: `count` ignores missing values while `size` counts every row.
  https://pandas.pydata.org/docs/reference/groupby.html

**Course materials**

- ME204 W02D01 on building dataframes from nested JSON and grouping them, and
  W02D04 on `.glob()` for listing files in a folder and `.get()` for reading
  dictionary keys that may be absent.